# 01 — Beijing Data Audit

Audit raw Beijing.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# Robust project root finder
# Notebook lives under: Project/notebooks/son_nb/
# ------------------------------------------------------------
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Project root. Hãy chạy notebook bên trong project."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAIN_OUTPUT = PROJECT_ROOT / "outputs"
SON_DIR = PROJECT_ROOT / "notebooks" / "son_nb"
SON_OUTPUT = SON_DIR / "output"

WORK_DATA = SON_OUTPUT / "data"
WORK_FIG = SON_OUTPUT / "figures"
WORK_TABLE = SON_OUTPUT / "tables"
WORK_PRED = SON_OUTPUT / "predictions"
WORK_MODEL = SON_OUTPUT / "models"

for p in [WORK_DATA, WORK_FIG, WORK_TABLE, WORK_PRED, WORK_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

BEIJING_RAW = RAW_DIR / "beijing" / "PRSA_data_2010.1.1-2014.12.31.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEIJING_RAW:", BEIJING_RAW)
print("Working output:", SON_OUTPUT)

PROJECT_ROOT: D:\Data-Science---SGU\Nam 4\PPNCKH\Project
BEIJING_RAW: D:\Data-Science---SGU\Nam 4\PPNCKH\Project\data\raw\beijing\PRSA_data_2010.1.1-2014.12.31.csv
Working output: D:\Data-Science---SGU\Nam 4\PPNCKH\Project\notebooks\son_nb\output


In [2]:
df = pd.read_csv(BEIJING_RAW)
print('shape =', df.shape)
print('columns =', df.columns.tolist())
display(df.head())
df.info()

shape = (43824, 13)
columns = ['No', 'year', 'month', 'day', 'hour', 'pm2.5', 'DEWP', 'TEMP', 'PRES', 'cbwd', 'Iws', 'Is', 'Ir']


,No,year,month,day,hour,pm2.5,DEWP,TEMP,PRES,cbwd,Iws,Is,Ir
0,1,2010,1,1,0,NaN,-21,-11.0,1021.0,NW,1.79,0,0
1,2,2010,1,1,1,NaN,-21,-12.0,1020.0,NW,4.92,0,0
2,3,2010,1,1,2,NaN,-21,-11.0,1019.0,NW,6.71,0,0
3,4,2010,1,1,3,NaN,-21,-14.0,1019.0,NW,9.84,0,0
4,5,2010,1,1,4,NaN,-20,-12.0,1018.0,NW,12.97,0,0


<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No      43824 non-null  int64  
 1   year    43824 non-null  int64  
 2   month   43824 non-null  int64  
 3   day     43824 non-null  int64  
 4   hour    43824 non-null  int64  
 5   pm2.5   41757 non-null  float64
 6   DEWP    43824 non-null  int64  
 7   TEMP    43824 non-null  float64
 8   PRES    43824 non-null  float64
 9   cbwd    43824 non-null  str    
 10  Iws     43824 non-null  float64
 11  Is      43824 non-null  int64  
 12  Ir      43824 non-null  int64  
dtypes: float64(4), int64(8), str(1)
memory usage: 4.4 MB


In [3]:
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('duplicate rows =', df.duplicated().sum())

,missing
pm2.5,2067
year,0
month,0
day,0
No,0
hour,0
DEWP,0
TEMP,0
PRES,0
cbwd,0


duplicate rows = 0


In [4]:
df['datetime'] = pd.to_datetime(df[['year','month','day','hour']])
df = df.sort_values('datetime').reset_index(drop=True)
expected = pd.date_range(df['datetime'].min(), df['datetime'].max(), freq='h')
missing_ts = expected.difference(df['datetime'])
print('start =', df['datetime'].min())
print('end =', df['datetime'].max())
print('duplicate datetime =', df['datetime'].duplicated().sum())
print('missing timestamps =', len(missing_ts))

start = 2010-01-01 00:00:00
end = 2014-12-31 23:00:00
duplicate datetime = 0
missing timestamps = 0


In [5]:
year_missing = df.groupby('year')['pm2.5'].agg(pm25_missing=lambda s:int(s.isna().sum()), total='size')
year_missing['missing_pct'] = 100*year_missing['pm25_missing']/year_missing['total']
display(year_missing)
year_missing.to_csv(WORK_TABLE/'beijing_pm25_missing_by_year.csv')

,pm25_missing,total,missing_pct
year,,,
2010,669,8760,7.636986
2011,728,8760,8.310502
2012,489,8784,5.566940
2013,82,8760,0.936073
2014,99,8760,1.130137


In [6]:
missing = df['pm2.5'].isna()
group = (missing != missing.shift()).cumsum()
gaps = (df.loc[missing].groupby(group[missing]).agg(start=('datetime','min'), end=('datetime','max'), hours=('datetime','size')).sort_values('hours', ascending=False))
display(gaps.head(20))
gaps.to_csv(WORK_TABLE/'beijing_pm25_missing_gaps.csv')

,start,end,hours
pm2.5,,,
41,2010-09-21 05:00:00,2010-09-27 15:00:00,155
287,2012-12-23 06:00:00,2012-12-28 12:00:00,127
163,2011-10-03 13:00:00,2011-10-07 15:00:00,99
61,2011-03-17 21:00:00,2011-03-21 15:00:00,91
43,2010-09-27 17:00:00,2010-09-30 20:00:00,76
185,2012-01-14 10:00:00,2012-01-17 11:00:00,74
259,2012-08-18 11:00:00,2012-08-21 12:00:00,74
27,2010-06-04 12:00:00,2010-06-07 11:00:00,72
35,2010-08-13 20:00:00,2010-08-16 15:00:00,68


In [7]:
quality = pd.DataFrame({'metric':['rows','columns','start','end','duplicate_datetime','missing_timestamps','pm25_missing','pm25_missing_pct'], 'value':[len(df),len(df.columns),str(df.datetime.min()),str(df.datetime.max()),int(df.datetime.duplicated().sum()),len(missing_ts),int(df['pm2.5'].isna().sum()),100*df['pm2.5'].isna().mean()]})
display(quality)
quality.to_csv(SON_OUTPUT/'beijing_data_quality_report.csv', index=False)

,metric,value
0,rows,43824
1,columns,14
2,start,2010-01-01 00:00:00
3,end,2014-12-31 23:00:00
4,duplicate_datetime,0
5,missing_timestamps,0
6,pm25_missing,2067
7,pm25_missing_pct,4.716594


### Gate 01
Chỉ sau khi audit rõ missing/gap/continuity mới sang preprocessing.

**Không nội suy PM2.5 toàn bộ ở bước này.**